In [1]:
import psutil
from tqdm import tqdm

def ram_usage():
    vm = psutil.virtual_memory()
    return f"RAM: {vm.used/(1024**3):.1f}/{vm.total/(1024**3):.1f} GB ({vm.percent:.0f}%)"
print(f"{ram_usage()}")

RAM: 13.4/15.8 GB (85%)


## 0 · Imports & config

In [10]:
import os, gc, sys, logging, json, re, shutil, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
from tqdm.auto import tqdm

logging.basicConfig(format='%(asctime)s  %(levelname)-8s  %(message)s',
                    level=logging.INFO, datefmt='%H:%M:%S')
log = logging.getLogger('preprocess')


ROOT_DIR      = Path(r'D:\mmd\data\raw')
REVIEW_PATH   = ROOT_DIR / 'review_Home_and_Kitchen.jsonl'
META_PATH     = ROOT_DIR / 'meta_Home_and_Kitchen.jsonl'

PROJECT_DIR      = Path(r'D:\mmd')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
CLEANED_DIR   = PROJECT_DIR / 'data' / 'cleaned'         
VOCAB_DIR     = PROJECT_DIR / 'data' / 'vocab'             
FIGURES_DIR   = PROJECT_DIR / 'outputs' / 'figures'
TEMP_DIR      = PROCESSED_DIR / '_temp_chunks'

META_OUTPUT_DIR   = PROCESSED_DIR / 'meta'
REVIEW_OUTPUT_DIR = PROCESSED_DIR / 'reviews'

for d in [
    PROCESSED_DIR,
    CLEANED_DIR,
    VOCAB_DIR,
    FIGURES_DIR,
    TEMP_DIR,
    META_OUTPUT_DIR,
    REVIEW_OUTPUT_DIR
]:
    d.mkdir(parents=True, exist_ok=True)
CHUNK_SIZE            = 500_000
MIN_USER_INTERACTIONS = 5       
MIN_ITEM_INTERACTIONS = 5       
MAX_SEQ_LEN           = 200    
RANDOM_SEED           = 42
np.random.seed(RANDOM_SEED)

print('=== CẤU HÌNH & THÔNG TIN FILE ===')
print(f'   Chunk size            : {CHUNK_SIZE:,} dòng/lần')
print(f'   Min user interactions : {MIN_USER_INTERACTIONS}')
print(f'   Min item interactions : {MIN_ITEM_INTERACTIONS}')
print(f'   Max seq len           : {MAX_SEQ_LEN}')
print(f'   Processed dir         : {PROCESSED_DIR}')
print(f'   Cleaned  dir          : {CLEANED_DIR}')

for label, path in [('Review JSONL', REVIEW_PATH), ('Meta JSONL', META_PATH)]:
    if path.exists():
        print(f'   {label}: {path.stat().st_size/(1024**3):.2f} GB')
    else:
        print(f'   {label}: KHÔNG TÌM THẤY tại {path}')

print(f'   {ram_usage()}')

=== CẤU HÌNH & THÔNG TIN FILE ===
   Chunk size            : 500,000 dòng/lần
   Min user interactions : 5
   Min item interactions : 5
   Max seq len           : 200
   Processed dir         : D:\mmd\data\processed
   Cleaned  dir          : D:\mmd\data\cleaned
   Review JSONL: 29.25 GB
   Meta JSONL: 10.98 GB
   RAM: 11.1/15.8 GB (70%)


## 1 · Load reviews (lazy)

In [11]:
gc.collect()

2525

In [12]:
import json
import gc
import psutil
import shutil
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from tqdm.auto import tqdm


import logging
logging.basicConfig(format="%(asctime)s  %(levelname)-8s  %(message)s",
                    level=logging.INFO, datefmt="%H:%M:%S")
log = logging.getLogger("preprocess")

log.info("Bước 1/3: Đọc JSONL và xả liên tục ra ổ cứng để ép RAM...")

TEMP_DIR = PROCESSED_DIR / "_temp_chunks"
TEMP_DIR.mkdir(parents=True, exist_ok=True)


for f in TEMP_DIR.glob("chunk_*.parquet"):
    f.unlink()

CHUNK_SIZE = 500_000
current_chunk = []
chunk_id = 0
total_raw = 0


with open(REVIEW_PATH, 'rt', encoding='utf-8') as f:
    pbar = tqdm(f, desc="Processing JSONL", unit=" lines")
    
    for line in pbar:
        total_raw += 1
        try:
            data = json.loads(line.strip())
        except json.JSONDecodeError:
            continue
            
       
        if not data.get("verified_purchase", False):
            continue
            
        u = data.get("user_id")
        it = data.get("parent_asin")
        r = data.get("rating")
        ts = data.get("timestamp")
        
        if not all([u, it, r, ts]):
            continue

        current_chunk.append({
            "user_id": u,
            "parent_asin": it,
            "rating": r,
            "timestamp": ts
        })
        
 
        if len(current_chunk) >= CHUNK_SIZE:
            df_chunk = pd.DataFrame(current_chunk)
            
        
            df_chunk = df_chunk.sort_values("timestamp", ascending=False)
            df_chunk = df_chunk.drop_duplicates(subset=["user_id", "parent_asin"], keep="first")
            
            chunk_path = TEMP_DIR / f"chunk_{chunk_id:04d}.parquet"
            df_chunk.to_parquet(chunk_path, index=False)
            
            chunk_id += 1
            current_chunk = [] 
            del df_chunk
            gc.collect()
            
            pbar.set_postfix({'Chunks': chunk_id, 'RAM': f"{psutil.virtual_memory().percent}%"})


if current_chunk:
    df_chunk = pd.DataFrame(current_chunk)
    df_chunk = df_chunk.sort_values("timestamp", ascending=False)
    df_chunk = df_chunk.drop_duplicates(subset=["user_id", "parent_asin"], keep="first")
    df_chunk.to_parquet(TEMP_DIR / f"chunk_{chunk_id:04d}.parquet", index=False)
    del df_chunk, current_chunk
    gc.collect()

log.info("Bước 2/3: Gộp luồng các chunk bằng ParquetWriter ...")

chunk_files = sorted(TEMP_DIR.glob("chunk_*.parquet"))
output_path = PROCESSED_DIR / "review_clean_temp.parquet"

writer = None
for chunk_file in tqdm(chunk_files, desc="Merging chunks"):
    table = pq.read_table(chunk_file)
    
    if writer is None:
        writer = pq.ParquetWriter(output_path, table.schema, compression='snappy')
        
    writer.write_table(table)
    del table
    gc.collect()

if writer:
    writer.close()


shutil.rmtree(TEMP_DIR)

log.info("Bước 3/3: Bắt đầu load file đã gộp vào RAM...")

df = pd.read_parquet(output_path, engine="pyarrow")

df['user_id']     = df['user_id'].astype(str)
df['parent_asin'] = df['parent_asin'].astype(str)
df['rating']      = pd.to_numeric(df['rating'], errors='coerce').astype('float32')
df['timestamp']   = pd.to_numeric(df['timestamp'], errors='coerce').astype('int64')
log.info(f"Đã load {len(df):,} dòng. Đang sắp xếp (Sort) timestamp in-place...")
df.sort_values("timestamp", ascending=False, inplace=True)
gc.collect()
log.info("Sort xong! Đang xóa trùng lặp (Drop Duplicates)...")

df.drop_duplicates(subset=["user_id", "parent_asin"], keep="first", inplace=True)
gc.collect()

log.info("Xóa trùng lặp xong! Đang format ngày tháng...")
df.reset_index(drop=True, inplace=True)

ts_unit = 'ms' if df['timestamp'].max() > 1_000_000_000_000 else 's'
df['dt'] = pd.to_datetime(df['timestamp'], unit=ts_unit)
log.info(f'Timestamp unit detected: {ts_unit}')
if output_path.exists():
    output_path.unlink()

print("\n=== KẾT QUẢ XỬ LÝ HOÀN TẤT ===")
print(f"  Tổng dòng đọc gốc : {total_raw:,}")
print(f"  Dòng giữ lại      : {len(df):,}")
print(f"  Date range        : {df['dt'].min().date()} → {df['dt'].max().date()}")


10:57:21  INFO      Bước 1/3: Đọc JSONL và xả liên tục ra ổ cứng để ép RAM...


Processing JSONL: 0 lines [00:00, ? lines/s]

11:05:35  INFO      Bước 2/3: Gộp luồng các chunk bằng ParquetWriter ...


Merging chunks:   0%|          | 0/126 [00:00<?, ?it/s]

11:06:04  INFO      Bước 3/3: Bắt đầu load file đã gộp vào RAM...
11:07:22  INFO      Đã load 62,201,078 dòng. Đang sắp xếp (Sort) timestamp in-place...
11:07:33  INFO      Sort xong! Đang xóa trùng lặp (Drop Duplicates)...
11:08:49  INFO      Xóa trùng lặp xong! Đang format ngày tháng...
11:08:53  INFO      Timestamp unit detected: ms



=== KẾT QUẢ XỬ LÝ HOÀN TẤT ===
  Tổng dòng đọc gốc : 67,409,944
  Dòng giữ lại      : 62,195,107
  Date range        : 2000-02-26 → 2023-09-13


## 2 · Clean Reviews
### 2.1 Load + dedup + filter verified

### 2.2 K-core filtering
Lặp lọc đến khi hội tụ: user ≥ 5 tương tác **và** item ≥ 5 tương tác.

In [13]:
import gc
from tqdm.auto import tqdm

def kcore_filter(df: pd.DataFrame, min_u: int, min_i: int) -> pd.DataFrame:
    """
    Lặp lọc user/item có ít tương tác đến khi kích thước không đổi.
    Thường hội tụ sau 3-5 vòng.
    """
    prev_len = -1
    iteration = 0
    
   
    pbar = tqdm(desc="K-core Filtering", unit=" iter")
    
    while len(df) != prev_len:
        prev_len = len(df)
        iteration += 1
        
        item_counts = df["parent_asin"].value_counts()
        df = df[df["parent_asin"].isin(item_counts[item_counts >= min_i].index)]
        
     
        user_counts = df["user_id"].value_counts()
        df = df[df["user_id"].isin(user_counts[user_counts >= min_u].index)]
        
        
        pbar.update(1)
        pbar.set_postfix({
            'Rows':  f'{len(df):,}',
            'Users': f'{len(user_counts):,}',
            'Items': f'{len(item_counts):,}',
            'RAM':   f'{psutil.virtual_memory().percent:.0f}%',
        })
      
        del item_counts, user_counts
        gc.collect()
        
    pbar.close()
    print(f"  Converged after {iteration} iterations.")
    return df.reset_index(drop=True)


print(f"Before k-core: {len(df):,} rows | "
      f"{df['user_id'].value_counts().size:,} users | "
      f"{df['parent_asin'].value_counts().size:,} items)")

df = kcore_filter(df, MIN_USER_INTERACTIONS, MIN_ITEM_INTERACTIONS)

print(f"\nFinal      : {len(df):,} rows | "
      f"{df['user_id'].value_counts().size:,} users | "
      f"{df['parent_asin'].value_counts().size:,} items)")
print(psutil.virtual_memory())
# =========================
# EXPORT FINAL REVIEW FILE
# =========================

REVIEW_OUTPUT_DIR = PROCESSED_DIR / "reviews"
REVIEW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_REVIEW_FILE = REVIEW_OUTPUT_DIR / "review_clean.parquet"

print("\nĐang lưu file review cuối cùng...")

df.to_parquet(
    FINAL_REVIEW_FILE,
    engine="pyarrow",
    compression="snappy",
    index=False
)

print(f"\nĐã lưu:")
print(FINAL_REVIEW_FILE)

print(
    f"Kích thước file: "
    f"{FINAL_REVIEW_FILE.stat().st_size / (1024**3):.2f} GB"
)

Before k-core: 62,195,107 rows | 22,468,224 users | 3,587,335 items)


K-core Filtering: 0 iter [00:00, ? iter/s]

  Converged after 8 iterations.

Final      : 24,930,015 rows | 2,712,338 users | 667,277 items)
svmem(total=16931676160, available=6742908928, percent=60.2, used=10188767232, free=6742908928)

Đang lưu file review cuối cùng...

Đã lưu:
D:\mmd\data\processed\reviews\review_clean.parquet
Kích thước file: 0.99 GB


## 3 · Clean Meta
Parse price, dedup `parent_asin`, filter chỉ giữ item có trong reviews.

In [23]:
import pandas as pd
import numpy as np
import re
import gc
from pathlib import Path

# =========================
# CONFIG
# =========================

log.info("Loading meta...")

META_COLS = [
    "parent_asin",
    "title",
    "main_category",
    "average_rating",
    "rating_number",
    "price",
    "store"
]

META_OUTPUT_DIR = PROCESSED_DIR / "meta"
META_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_META_FILE = META_OUTPUT_DIR / "meta_clean.parquet"

# =========================
# LOAD CLEANED PARQUET
# =========================

if FINAL_META_FILE.exists():

    print("Loading cleaned parquet meta...")

    df_meta = pd.read_parquet(
        FINAL_META_FILE,
        engine="pyarrow"
    )

# =========================
# LOAD RAW JSONL
# =========================

else:

    print("Loading raw jsonl meta in chunks...")

    chunks = []

    # chỉ lấy item có trong reviews
    valid_items = set(
        df["parent_asin"]
        .astype(str)
        .unique()
    )

    reader = pd.read_json(
        META_PATH,
        lines=True,
        chunksize=50000
    )

    for i, chunk in enumerate(reader):

        # =========================
        # KEEP NEEDED COLUMNS
        # =========================

        chunk = chunk.reindex(columns=META_COLS)

        # =========================
        # FILTER ITEMS
        # =========================

        chunk["parent_asin"] = (
            chunk["parent_asin"]
            .astype(str)
        )

        chunk = chunk.loc[
            chunk["parent_asin"].isin(valid_items)
        ].copy()

        # =========================
        # CLEAN DATATYPES
        # =========================

        chunk["average_rating"] = pd.to_numeric(
            chunk["average_rating"],
            errors="coerce"
        ).astype("float32")

        chunk["rating_number"] = pd.to_numeric(
            chunk["rating_number"],
            errors="coerce"
        ).fillna(0).astype("int32")

        # =========================
        # PARSE PRICE
        # =========================

        def parse_price(s):

            if pd.isna(s):
                return np.nan

            s = str(s).strip()

            if s in ("", "None", "null", "—"):
                return np.nan

            nums = re.findall(
                r"[\d]+(?:\.[\d]+)?",
                s
            )

            if not nums:
                return np.nan

            vals = [float(x) for x in nums]

            return round(sum(vals) / len(vals), 2)

        chunk["price_usd"] = (
            chunk["price"]
            .apply(parse_price)
            .astype("float32")
        )

        # =========================
        # APPEND
        # =========================

        chunks.append(chunk)

        # =========================
        # LOG
        # =========================

        if i % 10 == 0:

            print(
                f"Chunk {i:,} | "
                f"Rows: {len(chunk):,} | "
                f"RAM: {ram_usage()}"
            )

        del chunk
        gc.collect()

    # =========================
    # CONCAT
    # =========================

    print("\nCombining chunks...")

    df_meta = pd.concat(
        chunks,
        ignore_index=True
    )

    del chunks
    gc.collect()

    # =========================
    # REMOVE DUPLICATES
    # =========================

    df_meta = (
        df_meta
        .drop_duplicates(
            subset=["parent_asin"],
            keep="first"
        )
        .reset_index(drop=True)
    )

    # =========================
    # DROP RAW PRICE COLUMN
    # =========================

    df_meta = df_meta.drop(
        columns=["price"]
    )

    # =========================
    # SAVE PARQUET
    # =========================

    print("\nSaving parquet...")

    df_meta.to_parquet(
        FINAL_META_FILE,
        engine="pyarrow",
        compression="snappy",
        index=False
    )

# =========================
# FINAL INFO
# =========================

price_null = df_meta["price_usd"].isna().sum()

print("\n========== META SUMMARY ==========")

print(
    f"Rows              : {len(df_meta):,}"
)

print(
    f"Unique items      : "
    f"{df_meta['parent_asin'].nunique():,}"
)

print(
    f"Price valid       : "
    f"{len(df_meta)-price_null:,}"
)

print(
    f"Price NaN         : "
    f"{price_null:,}"
)

print(
    f"Price p99         : "
    f"${df_meta['price_usd'].quantile(0.99):.2f}"
)

print(
    f"RAM usage         : "
    f"{ram_usage()}"
)

print(
    f"Saved parquet     : "
    f"{FINAL_META_FILE}"
)

print(
    f"File size         : "
    f"{FINAL_META_FILE.stat().st_size / (1024**3):.2f} GB"
)

12:40:51  INFO      Loading meta...


Loading raw jsonl meta in chunks...
Chunk 0 | Rows: 18,158 | RAM: RAM: 10.2/15.8 GB (65%)
Chunk 10 | Rows: 12,541 | RAM: RAM: 10.5/15.8 GB (67%)
Chunk 20 | Rows: 10,458 | RAM: RAM: 10.6/15.8 GB (67%)
Chunk 30 | Rows: 9,570 | RAM: RAM: 10.6/15.8 GB (67%)
Chunk 40 | Rows: 9,153 | RAM: RAM: 11.2/15.8 GB (71%)
Chunk 50 | Rows: 8,339 | RAM: RAM: 11.1/15.8 GB (70%)
Chunk 60 | Rows: 8,394 | RAM: RAM: 11.1/15.8 GB (70%)
Chunk 70 | Rows: 15 | RAM: RAM: 11.1/15.8 GB (70%)

Combining chunks...

Saving parquet...

========== META SUMMARY ==========
Rows              : 667,277
Unique items      : 667,277
Price valid       : 343,160
Price NaN         : 324,117
Price p99         : $410.00
RAM usage         : RAM: 11.0/15.8 GB (70%)
Saved parquet     : D:\mmd\data\processed\meta\meta_clean.parquet
File size         : 0.06 GB


## 4 · Build Item Vocab
Map `parent_asin` (string) → `item_idx` (int) — cần thiết cho Item2Vec và GRU4Rec.

In [26]:
item_freq = df["parent_asin"].value_counts()

item2idx = {
    item: idx
    for idx, item in enumerate(item_freq.index)
}

idx2item = {
    idx: item
    for item, idx in item2idx.items()
}

N_ITEMS = len(item2idx)

# =========================
# USER VOCAB
# =========================

user_freq = df["user_id"].value_counts()

user2idx = {
    u: i
    for i, u in enumerate(user_freq.index)
}

N_USERS = len(user2idx)

# =========================
# MAP INDEX
# =========================

df["item_idx"] = df["parent_asin"].map(item2idx)

df["user_idx"] = df["user_id"].map(user2idx)

df_meta["item_idx"] = df_meta["parent_asin"].map(item2idx)

print(
    f"Vocab size : "
    f"{N_ITEMS:,} items | "
    f"{N_USERS:,} users"
)

# =========================
# SAVE VOCAB
# =========================

import json

vocab_dir = PROCESSED_DIR / "vocab"

vocab_dir.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    vocab_dir / "item2idx.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        item2idx,
        f,
        ensure_ascii=False
    )

with open(
    vocab_dir / "idx2item.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {str(k): v for k, v in idx2item.items()},
        f,
        ensure_ascii=False
    )

print(f"Vocab saved → {vocab_dir}")

Vocab size : 667,277 items | 2,712,338 users
Vocab saved → D:\mmd\data\processed\vocab


## 5 · Build User Sessions
Group by `user_id`, sort theo `timestamp` tăng dần → ra sequence `[item_idx_0, item_idx_1, ...]`.

In [27]:
print('Building sessions...')


df_sorted = df.sort_values(["user_idx", "timestamp"], ascending=True)


sessions = (
    df_sorted.groupby("user_idx", sort=False)["item_idx"]
    .apply(list)
    .reset_index()
    .rename(columns={"item_idx": "item_seq"})
)

sessions["item_seq"] = sessions["item_seq"].apply(
    lambda seq: seq[-MAX_SEQ_LEN:] if len(seq) > MAX_SEQ_LEN else seq
)
sessions["seq_len"] = sessions["item_seq"].apply(len)

print(f"Total sessions : {len(sessions):,}")
print(f"Sequence length:")
print(sessions["seq_len"].describe().to_string())
print(ram_usage())

Building sessions...
Total sessions : 2,712,338
Sequence length:
count    2.712338e+06
mean     9.189496e+00
std      6.927129e+00
min      5.000000e+00
25%      5.000000e+00
50%      7.000000e+00
75%      1.000000e+01
max      2.000000e+02
RAM: 12.9/15.8 GB (82%)


## 6 · Temporal Train / Val / Test Split
**Leave-One-Out (LOO) temporal:**  
- `test`  = item **cuối cùng** trong sequence mỗi user  
- `val`   = item **áp cuối**  
- `train` = tất cả phần còn lại  

Đây là chuẩn phổ biến nhất trong session-based recommendation.

In [28]:

sessions = sessions[sessions["seq_len"] >= 3].reset_index(drop=True)
print(f"Sessions with len>=3: {len(sessions):,}")


sessions["train_seq"] = sessions["item_seq"].apply(lambda s: s[:-2])
sessions["val_item"]  = sessions["item_seq"].apply(lambda s: s[-2])
sessions["test_item"] = sessions["item_seq"].apply(lambda s: s[-1])


total_interactions = sessions["seq_len"].sum()
train_interactions = sessions["train_seq"].apply(len).sum()
print(f"\nSplit summary:")
print(f"  train interactions : {train_interactions:,}  ({train_interactions/total_interactions*100:.1f}%)")
print(f"  val  interactions  : {len(sessions):,}  ({len(sessions)/total_interactions*100:.1f}%)")
print(f"  test interactions  : {len(sessions):,}  ({len(sessions)/total_interactions*100:.1f}%)")
print(f"  total              : {total_interactions:,}")

Sessions with len>=3: 2,712,338

Split summary:
  train interactions : 19,500,343  (78.2%)
  val  interactions  : 2,712,338  (10.9%)
  test interactions  : 2,712,338  (10.9%)
  total              : 24,925,019


## 7 · Save outputs

In [30]:

clean_cols = ["user_idx", "user_id", "item_idx", "parent_asin",
              "rating", "timestamp", "dt"]
df[clean_cols].to_parquet(CLEANED_DIR / "reviews_clean.parquet",
                           index=False, engine="pyarrow")
print(f"Saved reviews_clean.parquet  ({len(df):,} rows)")


df_meta.to_parquet(CLEANED_DIR / "meta_clean.parquet",
                   index=False, engine="pyarrow")
print(f"Saved meta_clean.parquet     ({len(df_meta):,} rows)")

import pickle
sessions_save = sessions[["user_idx", "train_seq", "val_item",
                           "test_item", "seq_len"]].copy()


with open(CLEANED_DIR / "sessions.pkl", "wb") as f:
    pickle.dump(sessions_save, f)
print(f"Saved sessions.pkl           ({len(sessions_save):,} users)")


records = []
for row in tqdm(sessions_save.itertuples(), total=len(sessions_save), desc="Flatten"):
    for pos, item in enumerate(row.train_seq):
        records.append((row.user_idx, pos, item, "train"))
    records.append((row.user_idx, len(row.train_seq), row.val_item, "val"))
    records.append((row.user_idx, len(row.train_seq)+1, row.test_item, "test"))

df_flat = pd.DataFrame(records, columns=["user_idx", "position", "item_idx", "split"])
df_flat.to_parquet(CLEANED_DIR / "interactions_flat.parquet",
                   index=False, engine="pyarrow")
print(f"Saved interactions_flat.parquet  ({len(df_flat):,} rows)")
del records, df_flat; gc.collect()

print("\n" + "="*50)
print("  PREPROCESSING COMPLETE")
print("="*50)
print(f"  Users          : {N_USERS:>10,}")
print(f"  Items          : {N_ITEMS:>10,}")
print(f"  Interactions   : {len(df):>10,}")
print(f"  Sessions       : {len(sessions_save):>10,}")
print(f"  Avg seq len    : {sessions_save['seq_len'].mean():>10.2f}")
print(f"  Sparsity       : {1 - len(df)/(N_USERS*N_ITEMS):>10.6f}")
print("="*50)
print(ram_usage())

Saved reviews_clean.parquet  (24,930,015 rows)
Saved meta_clean.parquet     (667,277 rows)
Saved sessions.pkl           (2,712,338 users)


Flatten:   0%|          | 0/2712338 [00:00<?, ?it/s]

Saved interactions_flat.parquet  (24,925,019 rows)

  PREPROCESSING COMPLETE
  Users          :  2,712,338
  Items          :    667,277
  Interactions   : 24,930,015
  Sessions       :  2,712,338
  Avg seq len    :       9.19
  Sparsity       :   0.999986
RAM: 12.1/15.8 GB (77%)
